In [1]:
import os
import json
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from tqdm import tqdm

PROJECT_ROOT = ".."
INPUT_BASE = os.path.join(PROJECT_ROOT, "data", "final")
OUTPUT_BASE = os.path.join(PROJECT_ROOT, "data", "interpolated_json")
CATEGORIES = ["bench_press", "pull_up", "push_up", "squat"]
VISIBILITY_THRESHOLD = 0.5

In [ ]:
def interpolate_skeletal_data(json_data, threshold=0.5):
    # Determine if data is a list or a dict with a nested list
    if isinstance(json_data, dict):
        # Look for the largest list in the dictionary
        list_key = next((k for k, v in json_data.items() if isinstance(v, list)), None)
        if list_key:
            frames = json_data[list_key]
        else:
            raise ValueError("No skeletal list found in JSON structure.")
    else:
        frames = json_data

    df = pd.DataFrame(frames)
    total_frames = len(df)
    
    for i in range(33):
        x_col, y_col, z_col = f'lm{i}_x', f'lm{i}_y', f'lm{i}_z'
        v_col = next((c for c in [f'lm{i}_v', f'lm{i}_vis', f'lm{i}_visibility'] if c in df.columns), None)
        
        if v_col:
            mask = df[v_col] < threshold
            df.loc[mask, [x_col, y_col, z_col]] = np.nan
        
        time_indices = np.arange(total_frames)
        for axis in [x_col, y_col, z_col]:
            if axis in df.columns:
                valid_idx = time_indices[~df[axis].isna()]
                valid_values = df[axis].dropna()
                if len(valid_idx) > 3:
                    cs = CubicSpline(valid_idx, valid_values, bc_type='natural')
                    df[axis] = cs(time_indices)
    
    df = df.ffill().bfill()
    return df.to_dict(orient='records')

In [3]:
print("Starting CoreSet Interpolation Audit...")
stats = {cat: {'input': 0, 'output': 0} for cat in CATEGORIES}

for category in CATEGORIES:
    input_folder = os.path.join(INPUT_BASE, category)
    output_folder = os.path.join(OUTPUT_BASE, category)
    os.makedirs(output_folder, exist_ok=True)
    
    if not os.path.exists(input_folder): continue
    files = [f for f in os.listdir(input_folder) if f.endswith('.json')]
    stats[category]['input'] = len(files)
    
    for filename in tqdm(files, desc=f"Processing {category}", leave=False):
        with open(os.path.join(input_folder, filename), 'r') as f:
            try:
                raw_data = json.load(f)
                
                # Dynamic check for frame list length
                if isinstance(raw_data, dict):
                    list_key = next((k for k, v in raw_data.items() if isinstance(v, list)), None)
                    original_len = len(raw_data[list_key]) if list_key else 0
                else:
                    original_len = len(raw_data)
                
                clean_data = interpolate_skeletal_data(raw_data, VISIBILITY_THRESHOLD)
                
                if len(clean_data) == original_len:
                    with open(os.path.join(output_folder, filename), 'w') as out_f:
                        json.dump(clean_data, out_f)
                    stats[category]['output'] += 1
            except Exception as e:
                pass

Starting CoreSet Interpolation Audit...


In [4]:
# --- FINAL REPORT ---
print("\n" + "="*60)
print(f"{'CORESET CATEGORY AUDIT REPORT':^60}")
print("="*60)
print(f"{'Category':<20} | {'Input Files':<15} | {'Output (Success)':<15}")
print("-" * 60)
total_in, total_out = 0, 0
for cat, counts in stats.items():
    print(f"{cat.replace('_', ' ').title():<20} | {counts['input']:<15} | {counts['output']:<15}")
    total_in += counts['input']
    total_out += counts['output']
print("-" * 60)
print(f"{'TOTAL':<20} | {total_in:<15} | {total_out:<15}")
print("="*60)


               CORESET CATEGORY AUDIT REPORT                
Category             | Input Files     | Output (Success)
------------------------------------------------------------
Bench Press          | 250             | 250            
Pull Up              | 251             | 251            
Push Up              | 250             | 250            
Squat                | 250             | 250            
------------------------------------------------------------
TOTAL                | 1001            | 1001           
